# Test.csv 3-Model Translation Only (Unsloth)

- Purpose: run EN->KO generation only (no COMET/metrics).
- Models: `alwaysgood/qwen3-it`, `alwaysgood/qwen35-it`, `alwaysgood/gemma4-it`.
- Env target: `unsloth` + `transformers==5.5.4`.
- Output: `translations.csv`, `translations_by_model_wide.csv`.


In [ ]:
# Runtime bootstrap for translation kernel
import subprocess, sys

def pip_install(args, optional=False):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q'] + list(args)
    print('running:', ' '.join(cmd))
    try:
        subprocess.check_call(cmd)
        return True
    except Exception as e:
        if optional:
            print(f'[WARN] optional install failed: {args} -> {type(e).__name__}: {e}')
            return False
        raise

pip_install(['unsloth','unsloth-zoo','datasets','trl','hydra-core','omegaconf','sentencepiece','numpy'])
pip_install(['pandas','tqdm','pyyaml>=6.0.2'])
XFORMERS_INSTALL_OK = pip_install(['xformers'], optional=True)
print('xformers_install_ok=', XFORMERS_INSTALL_OK)
pip_install(['transformers==5.5.4','trl>=0.15.0','--no-deps'])
print('bootstrap install done. Restart kernel, then Run All.')


In [ ]:
import gc, os, re, shutil, warnings, importlib.util, hashlib, json as pyjson
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoTokenizer

warnings.filterwarnings('ignore')
if transformers.__version__ != '5.5.4':
    raise RuntimeError(f'Expected transformers==5.5.4, got {transformers.__version__}')
if importlib.util.find_spec('unsloth') is None:
    raise RuntimeError('unsloth is required in this notebook.')
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('cuda_available', torch.cuda.is_available())


In [ ]:
# Paths + run config
def _first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for base in [start, *start.parents, Path('/workspace/scp_stage3_it'), Path('/workspace')]:
        prompt_yaml = base / 'configs' / 'prompts' / 'translation_dynamic_5.yaml'
        sft_yaml = base / 'configs' / 'preprocessing' / 'sft_translation.yaml'
        if prompt_yaml.exists() and sft_yaml.exists():
            return base.resolve()
    return start

WORK_ROOT = Path(os.environ.get('WORK_ROOT', '')).expanduser().resolve() if os.environ.get('WORK_ROOT','').strip() else _find_repo_root(Path.cwd())
INPUT_CSV = Path(os.environ.get('INPUT_CSV', str(_first_existing([WORK_ROOT/'data'/'eval'/'test.csv', WORK_ROOT/'data'/'test.csv', WORK_ROOT/'test.csv']) or (WORK_ROOT/'data'/'eval'/'test.csv')))).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get('OUTPUT_DIR', str(WORK_ROOT/'outputs'/'testcsv_3model'))).expanduser().resolve()
HF_CACHE_ROOT = Path(os.environ.get('HF_CACHE_ROOT', str(OUTPUT_DIR/'hf_cache'))).expanduser().resolve()
PROMPT_CONFIG_PATH = Path(os.environ.get('PROMPT_CONFIG_PATH', str(WORK_ROOT/'configs'/'prompts'/'translation_dynamic_5.yaml'))).expanduser().resolve()
SFT_CONFIG_PATH = Path(os.environ.get('SFT_CONFIG_PATH', str(WORK_ROOT/'configs'/'preprocessing'/'sft_translation.yaml'))).expanduser().resolve()
SOURCE_LANG_ISO = os.environ.get('SOURCE_LANG_ISO', 'en').strip().lower()
TARGET_LANG_ISO = os.environ.get('TARGET_LANG_ISO', 'ko').strip().lower()
TEMPLATE_SEED = int(os.environ.get('TEMPLATE_SEED', '42'))
_fti = os.environ.get('FIXED_TEMPLATE_INDEX', '').strip()
FIXED_TEMPLATE_INDEX = int(_fti) if _fti else None
MODELS = ['alwaysgood/qwen3-it','alwaysgood/qwen35-it','alwaysgood/gemma4-it']
MAX_NEW_TOKENS = int(os.environ.get('MAX_NEW_TOKENS', '512'))
BATCH_SIZE_QWEN = int(os.environ.get('BATCH_SIZE_QWEN', '8'))
BATCH_SIZE_GEMMA = int(os.environ.get('BATCH_SIZE_GEMMA', '6'))
SORT_BY_INPUT_LENGTH = True
USE_UNSLOTH = True
UNSLOTH_MAX_SEQ_LENGTH = int(os.environ.get('UNSLOTH_MAX_SEQ_LENGTH', '4096'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print('WORK_ROOT', WORK_ROOT)
print('INPUT_CSV', INPUT_CSV)
print('OUTPUT_DIR', OUTPUT_DIR)
print('MAX_NEW_TOKENS (generated only)', MAX_NEW_TOKENS)


In [ ]:
# Load input CSV
if not INPUT_CSV.exists():
    raise FileNotFoundError(f'Input CSV not found: {INPUT_CSV}')
raw_df = pd.read_csv(INPUT_CSV, encoding='utf-8-sig').copy()
def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f'Missing required column. candidates={candidates}, available={list(df.columns)}')
    return None
id_col = pick_col(raw_df, ['ID','\ufeffID','id','row_id'])
source_col = pick_col(raw_df, ['Source_En','source_text','source_en'])
target_col = pick_col(raw_df, ['Target_Ko','target_text','reference_text','target_ko'])
src_lang_col = pick_col(raw_df, ['source_lang_iso','src_lang_iso','source_lang','Source_Lang'], required=False)
tgt_lang_col = pick_col(raw_df, ['target_lang_iso','tgt_lang_iso','target_lang','Target_Lang'], required=False)
eval_df = raw_df[[id_col, source_col, target_col]].copy().rename(columns={id_col:'row_id', source_col:'source_text', target_col:'reference_text'})
eval_df['row_id'] = pd.to_numeric(eval_df['row_id'], errors='coerce')
eval_df = eval_df.dropna(subset=['row_id']).copy()
eval_df['row_id'] = eval_df['row_id'].astype(int)
eval_df['source_text'] = eval_df['source_text'].fillna('').astype(str)
eval_df['reference_text'] = eval_df['reference_text'].fillna('').astype(str)
eval_df['source_lang_iso'] = (raw_df.loc[eval_df.index, src_lang_col].fillna(SOURCE_LANG_ISO).astype(str).str.strip().str.lower() if src_lang_col is not None else SOURCE_LANG_ISO)
eval_df['target_lang_iso'] = (raw_df.loc[eval_df.index, tgt_lang_col].fillna(TARGET_LANG_ISO).astype(str).str.strip().str.lower() if tgt_lang_col is not None else TARGET_LANG_ISO)
eval_df['source_lang_iso'] = eval_df['source_lang_iso'].replace('', SOURCE_LANG_ISO)
eval_df['target_lang_iso'] = eval_df['target_lang_iso'].replace('', TARGET_LANG_ISO)
eval_df = eval_df.sort_values('row_id').reset_index(drop=True)
print('rows', len(eval_df))
display(eval_df.head(3))


In [ ]:
# Helpers + Unsloth loader
DEFAULT_LANG_NAME_MAP = {'en':'English','ko':'Korean','ja':'Japanese','zh':'Chinese'}
DEFAULT_LANG_LOCALE_MAP = {'en':'en-US','ko':'ko-KR','ja':'ja-JP','zh':'zh-CN'}
def slugify_model(model_id: str) -> str:
    return re.sub(r'[^A-Za-z0-9._-]+', '_', model_id)
def resolve_batch_size(model_id: str) -> int:
    return int(BATCH_SIZE_GEMMA) if 'gemma' in model_id.lower() else int(BATCH_SIZE_QWEN)
def _load_structured_file(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f'Config file not found: {path}')
    text = path.read_text(encoding='utf-8')
    if path.suffix.lower() == '.json':
        data = pyjson.loads(text)
        if not isinstance(data, dict):
            raise ValueError(f'JSON config must be object: {path}')
        return data
    import yaml
    data = yaml.safe_load(text)
    if not isinstance(data, dict):
        raise ValueError(f'YAML config must be mapping: {path}')
    return data
def _as_dict(x):
    return x if isinstance(x, dict) else {}
def load_prompt_assets(prompt_config_path: Path, sft_config_path: Path):
    prompt_cfg = _load_structured_file(prompt_config_path)
    templates = prompt_cfg.get('templates')
    if not isinstance(templates, list) or not templates:
        templates = _as_dict(prompt_cfg.get('prompts')).get('templates')
    if not isinstance(templates, list) or not templates:
        raise ValueError('Prompt config needs templates or prompts.templates')
    sft_raw = _load_structured_file(sft_config_path)
    sft_cfg = _as_dict(sft_raw.get('sft')) or _as_dict(_as_dict(sft_raw.get('preprocessing')).get('sft'))
    response_prefix = str(sft_cfg.get('response_prefix', 'response: '))
    lang_name_map = dict(DEFAULT_LANG_NAME_MAP); lang_name_map.update({str(k).strip().lower():str(v) for k,v in _as_dict(sft_cfg.get('lang_name_map')).items()})
    lang_locale_map = dict(DEFAULT_LANG_LOCALE_MAP); lang_locale_map.update({str(k).strip().lower():str(v) for k,v in _as_dict(sft_cfg.get('lang_locale_map')).items()})
    return [str(t) for t in templates], response_prefix, lang_name_map, lang_locale_map
def stable_template_index(sample_key: str, template_count: int, seed: int) -> int:
    key = f'{seed}|{sample_key}'.encode('utf-8')
    return int.from_bytes(hashlib.blake2b(key, digest_size=8).digest(), 'big') % template_count
def resolve_template_for_row(prompt_templates, row_id: str, template_seed: int, fixed_template_index):
    if fixed_template_index is not None:
        if fixed_template_index < 0 or fixed_template_index >= len(prompt_templates):
            raise ValueError('FIXED_TEMPLATE_INDEX out of range')
        return prompt_templates[fixed_template_index], fixed_template_index
    idx = stable_template_index(str(row_id), len(prompt_templates), int(template_seed))
    return prompt_templates[idx], idx
def resolve_lang_context(src_iso, tgt_iso, name_map, locale_map):
    s = str(src_iso or SOURCE_LANG_ISO).strip().lower() or SOURCE_LANG_ISO
    t = str(tgt_iso or TARGET_LANG_ISO).strip().lower() or TARGET_LANG_ISO
    return s, t, name_map.get(s, s.upper()), name_map.get(t, t.upper()), locale_map.get(s, s), locale_map.get(t, t)
def render_prompt(source_text, prompt_template, response_prefix, src_lang_name, tgt_lang_name, src_locale, tgt_locale):
    prompt = prompt_template.format(src_lang_name=src_lang_name, tgt_lang_name=tgt_lang_name, src_locale=src_locale, tgt_locale=tgt_locale, src=source_text)
    return f'{prompt}\n{response_prefix}'
def clean_hypothesis(text, response_prefix):
    out = str(text or '').strip()
    rp = str(response_prefix or '').strip()
    if rp and rp in out:
        out = out.rsplit(rp, 1)[-1].strip()
    for p in (rp, '<KO>', '<ko>', '[KO]', 'KO:', '번역:', 'Translation:'):
        if p and out.startswith(p):
            out = out[len(p):].strip()
    return out
def _load_with_unsloth(model_id: str):
    from unsloth import FastLanguageModel, FastVisionModel
    errors=[]
    for backend, cls in (("language", FastLanguageModel),("vision", FastVisionModel)):
        try:
            dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else None
            model, tokenizer = cls.from_pretrained(model_name=model_id, max_seq_length=int(UNSLOTH_MAX_SEQ_LENGTH), dtype=dtype, load_in_4bit=False)
            return model, tokenizer, backend, dtype
        except Exception as e:
            errors.append(f'{backend}: {type(e).__name__}: {e}')
    raise RuntimeError('Unsloth load failed for '+model_id+'\n'+'\n'.join(errors))
def load_model_and_tokenizer(model_id: str, cache_dir: Path):
    cache_dir.mkdir(parents=True, exist_ok=True)
    os.environ['HF_HOME']=str(cache_dir); os.environ['TRANSFORMERS_CACHE']=str(cache_dir); os.environ['HUGGINGFACE_HUB_CACHE']=str(cache_dir)
    _ = AutoConfig.from_pretrained(model_id, trust_remote_code=True, cache_dir=str(cache_dir))
    model, tokenizer, backend, dtype = _load_with_unsloth(model_id)
    print(f'  loader=unsloth/{backend} dtype={dtype}')
    if backend == 'language':
        try:
            from unsloth import FastLanguageModel
            FastLanguageModel.for_inference(model)
            print('  unsloth_for_inference=True')
        except Exception as e:
            print(f'[WARN] unsloth for_inference failed: {type(e).__name__}: {e}')
    tok = getattr(tokenizer, 'tokenizer', tokenizer)
    if tok.pad_token_id is None: tok.pad_token = tok.eos_token
    tok.padding_side='left'
    if hasattr(model,'config'): model.config.use_cache=True
    if hasattr(model,'generation_config'): model.generation_config.use_cache=True
    model.eval()
    return model, tok, next(model.parameters()).device
def _build_gen_kwargs(tokenizer):
    return {'max_new_tokens':int(MAX_NEW_TOKENS),'do_sample':False,'pad_token_id':tokenizer.pad_token_id,'eos_token_id':tokenizer.eos_token_id,'use_cache':True,'num_beams':1}
def _get_context_limit(model, tokenizer):
    m = getattr(getattr(model,'config',None),'max_position_embeddings',None)
    if isinstance(m,int) and m>0: return m
    t = getattr(tokenizer,'model_max_length',None)
    return t if isinstance(t,int) and 0<t<1_000_000 else None
def generate_for_rows(model, tokenizer, device, rows_df, model_id, prompt_templates, response_prefix, lang_name_map, lang_locale_map):
    batch_size = max(1, resolve_batch_size(model_id))
    work = rows_df.reset_index(drop=True).copy(); work['_orig_pos']=work.index.astype(int)
    rendered=[]
    for _,r in work.iterrows():
        s_iso,t_iso,s_name,t_name,s_loc,t_loc = resolve_lang_context(r.get('source_lang_iso'), r.get('target_lang_iso'), lang_name_map, lang_locale_map)
        tmpl, idx = resolve_template_for_row(prompt_templates, str(int(r['row_id'])), TEMPLATE_SEED, FIXED_TEMPLATE_INDEX)
        rendered.append({'_orig_pos':int(r['_orig_pos']),'row_id':int(r['row_id']),'source_text':str(r['source_text']),'reference_text':str(r['reference_text']),'source_lang_iso':s_iso,'target_lang_iso':t_iso,'template_index':int(idx),'prompt':render_prompt(str(r['source_text']), tmpl, response_prefix, s_name, t_name, s_loc, t_loc)})
    if SORT_BY_INPUT_LENGTH: rendered.sort(key=lambda x:(len(x['source_text']),x['_orig_pos']))
    results=[]; total=len(rendered); limit=_get_context_limit(model, tokenizer)
    print(f'infer_batch_size={batch_size} sort_by_length={SORT_BY_INPUT_LENGTH} context_limit={limit}')
    for start in range(0,total,batch_size):
        chunk = rendered[start:start+batch_size]
        try:
            enc = tokenizer([x['prompt'] for x in chunk], return_tensors='pt', add_special_tokens=False, padding=True, truncation=False, pad_to_multiple_of=8)
            input_ids = enc['input_ids'].to(device); attn = enc['attention_mask'].to(device); batch_input_len = int(input_ids.shape[1])
            kwargs = _build_gen_kwargs(tokenizer)
            if limit is not None:
                allowed = int((limit - attn.sum(dim=1)).min().item())
                if allowed <= 0: raise RuntimeError(f'input exceeds context limit in batch: limit={limit}')
                kwargs['max_new_tokens'] = min(kwargs['max_new_tokens'], allowed)
            with torch.inference_mode():
                out = model.generate(input_ids=input_ids, attention_mask=attn, **kwargs)
            for i,row in enumerate(chunk):
                hyp = clean_hypothesis(tokenizer.decode(out[i][batch_input_len:], skip_special_tokens=True), response_prefix)
                results.append({'_orig_pos':row['_orig_pos'],'row_id':row['row_id'],'source_text':row['source_text'],'reference_text':row['reference_text'],'source_lang_iso':row['source_lang_iso'],'target_lang_iso':row['target_lang_iso'],'template_index':str(row['template_index']),'hypothesis':hyp,'error':''})
        except Exception as e:
            err=f'{type(e).__name__}: {e}'
            if 'out of memory' in str(e).lower() and len(chunk)>1 and torch.cuda.is_available():
                torch.cuda.empty_cache()
            for row in chunk:
                results.append({'_orig_pos':row['_orig_pos'],'row_id':row['row_id'],'source_text':row['source_text'],'reference_text':row['reference_text'],'source_lang_iso':row['source_lang_iso'],'target_lang_iso':row['target_lang_iso'],'template_index':str(row['template_index']),'hypothesis':'','error':err})
        done=min(start+len(chunk),total)
        if done%20==0 or done==total: print(f'progress {done}/{total}')
    results.sort(key=lambda x:x.get('_orig_pos',0))
    for r in results: r.pop('_orig_pos',None)
    return results
def cleanup_model_cache(cache_dir, model=None, tokenizer=None):
    del model; del tokenizer; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache(); torch.cuda.synchronize()
    if cache_dir.exists(): shutil.rmtree(cache_dir, ignore_errors=True)

PROMPT_TEMPLATES, RESPONSE_PREFIX, LANG_NAME_MAP, LANG_LOCALE_MAP = load_prompt_assets(PROMPT_CONFIG_PATH, SFT_CONFIG_PATH)
print('prompt_templates_count', len(PROMPT_TEMPLATES))
print('response_prefix', repr(RESPONSE_PREFIX))


In [ ]:
# Sequential generation
all_result_frames=[]
expected_rows=len(eval_df)
expected_row_ids=set(eval_df['row_id'].astype(int).tolist())
for model_id in MODELS:
    slug=slugify_model(model_id)
    cache_dir=HF_CACHE_ROOT/slug
    per_model_csv=OUTPUT_DIR/f'{slug}__translations.csv'
    print('\n'+'='*100)
    print('[MODEL]', model_id)
    print('='*100)
    if per_model_csv.exists():
        existing=pd.read_csv(per_model_csv, encoding='utf-8-sig')
        req=['row_id','model_name','source_text','reference_text','source_lang_iso','target_lang_iso','template_index','hypothesis','error']
        valid=False
        if set(req).issubset(set(existing.columns)) and len(existing)==expected_rows:
            ex=existing[req].copy(); ex['row_id']=pd.to_numeric(ex['row_id'], errors='coerce'); ex=ex.dropna(subset=['row_id']).copy(); ex['row_id']=ex['row_id'].astype(int)
            if len(ex)==expected_rows:
                valid=(set(ex['row_id'])==expected_row_ids and ex['row_id'].nunique()==expected_rows and set(ex['model_name'].astype(str).dropna().unique().tolist())=={model_id})
        if valid:
            print('skip existing (validated):', per_model_csv)
            all_result_frames.append(ex[req].copy())
            continue
        print('existing output invalid; regenerating:', per_model_csv)
    model=tokenizer=None
    try:
        model, tokenizer, device = load_model_and_tokenizer(model_id, cache_dir)
        rows = generate_for_rows(model, tokenizer, device, eval_df, model_id, PROMPT_TEMPLATES, RESPONSE_PREFIX, LANG_NAME_MAP, LANG_LOCALE_MAP)
        per_df=pd.DataFrame(rows); per_df['model_name']=model_id
        per_df=per_df[['row_id','model_name','source_text','reference_text','source_lang_iso','target_lang_iso','template_index','hypothesis','error']]
        per_df.to_csv(per_model_csv, index=False, encoding='utf-8-sig')
        print('saved:', per_model_csv)
        all_result_frames.append(per_df.copy())
    finally:
        cleanup_model_cache(cache_dir, model=model, tokenizer=tokenizer)
print('\nAll model runs completed.')


In [ ]:
# Merge outputs
translations_path=OUTPUT_DIR/'translations.csv'
if not all_result_frames:
    raise RuntimeError('No results collected from generation step.')
translations_df=pd.concat(all_result_frames, axis=0, ignore_index=True)
translations_df=translations_df[['row_id','model_name','source_text','reference_text','source_lang_iso','target_lang_iso','template_index','hypothesis','error']]
translations_df=translations_df.sort_values(['row_id','model_name']).reset_index(drop=True)
translations_df.to_csv(translations_path, index=False, encoding='utf-8-sig')
print('saved:', translations_path)
print('total_rows:', len(translations_df))
display(translations_df.groupby('model_name')['row_id'].count().sort_values(ascending=False))
display(translations_df.assign(has_error=translations_df['error'].fillna('').str.strip().ne('')).groupby('model_name')['has_error'].sum())
wide_df=(translations_df.pivot_table(index=['row_id','source_text','reference_text'], columns='model_name', values='hypothesis', aggfunc='first').reset_index())
wide_df.columns.name=None
wide_df=wide_df.sort_values('row_id').reset_index(drop=True)
wide_path=OUTPUT_DIR/'translations_by_model_wide.csv'
wide_df.to_csv(wide_path, index=False, encoding='utf-8-sig')
print('saved:', wide_path)
display(wide_df.head(5))
